# 01 — Data Understanding

Goal: understand the structure and quality of the UNSW-NB15 training data.

Initial questions:
- What does one row represent?
- How many rows and columns are present?
- What columns are available?
- How has pandas interpreted their data types?

The official testing file is reserved for final evaluation.

In [1]:
# Load tools for paths, tables, and checking the Python environment.
import sys
from pathlib import Path

import pandas as pd

# Confirm Jupyter is using this project's environment.
print("Python:", sys.executable)
print("Working directory:", Path.cwd())
print("pandas version:", pd.__version__)

Python: C:\Code\fourthyear\resume_projects\network_intrusion_detection_system\.venv\Scripts\python.exe
Working directory: C:\Code\fourthyear\resume_projects\network_intrusion_detection_system\notebooks
pandas version: 3.0.5


In [2]:
# From notebooks/, go up one folder and locate the training CSV.
project_root = Path.cwd().parent
training_path = project_root / "data" / "raw" / "UNSW_NB15_training-set.csv"

print("Training file:", training_path)
print("File exists:", training_path.is_file())

Training file: C:\Code\fourthyear\resume_projects\network_intrusion_detection_system\data\raw\UNSW_NB15_training-set.csv
File exists: True


In [3]:
# Read the CSV into a pandas table; train_df holds the full training file.
train_df = pd.read_csv(training_path)

# Show (number of rows, number of columns).
train_df.shape

(175341, 45)

In [4]:
# Preview five records to see how the columns are stored.
train_df.head()

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


In [5]:
# List every column name, including the ID and target annotations.
train_df.columns.tolist()

['id',
 'dur',
 'proto',
 'service',
 'state',
 'spkts',
 'dpkts',
 'sbytes',
 'dbytes',
 'rate',
 'sttl',
 'dttl',
 'sload',
 'dload',
 'sloss',
 'dloss',
 'sinpkt',
 'dinpkt',
 'sjit',
 'djit',
 'swin',
 'stcpb',
 'dtcpb',
 'dwin',
 'tcprtt',
 'synack',
 'ackdat',
 'smean',
 'dmean',
 'trans_depth',
 'response_body_len',
 'ct_srv_src',
 'ct_state_ttl',
 'ct_dst_ltm',
 'ct_src_dport_ltm',
 'ct_dst_sport_ltm',
 'ct_dst_src_ltm',
 'is_ftp_login',
 'ct_ftp_cmd',
 'ct_flw_http_mthd',
 'ct_src_ltm',
 'ct_srv_dst',
 'is_sm_ips_ports',
 'attack_cat',
 'label']

In [6]:
# Check column types and non-missing counts before choosing preprocessing.
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 175341 entries, 0 to 175340
Data columns (total 45 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 175341 non-null  int64  
 1   dur                175341 non-null  float64
 2   proto              175341 non-null  str    
 3   service            175341 non-null  str    
 4   state              175341 non-null  str    
 5   spkts              175341 non-null  int64  
 6   dpkts              175341 non-null  int64  
 7   sbytes             175341 non-null  int64  
 8   dbytes             175341 non-null  int64  
 9   rate               175341 non-null  float64
 10  sttl               175341 non-null  int64  
 11  dttl               175341 non-null  int64  
 12  sload              175341 non-null  float64
 13  dload              175341 non-null  float64
 14  sloss              175341 non-null  int64  
 15  dloss              175341 non-null  int64  
 16  sinpkt       

In [7]:
# Count normal (0) and attack (1) labels, including missing values if present.
label_counts = train_df["label"].value_counts(dropna=False).sort_index()

# Convert counts to percentages to inspect class imbalance.
label_summary = pd.DataFrame({
    "count": label_counts,
    "percentage": label_counts / len(train_df) * 100,
})

label_summary

,count,percentage
label,,
0,56000,31.937767
1,119341,68.062233


In [8]:
# Check how attack categories map to labels; attack_cat reveals our target.
pd.crosstab(
    train_df["attack_cat"],
    train_df["label"],
    dropna=False,
)

label,0,1
attack_cat,,
Analysis,0,2000
Backdoor,0,1746
DoS,0,12264
Exploits,0,33393
Fuzzers,0,18184
Generic,0,40000
Normal,56000,0
Reconnaissance,0,10491
Shellcode,0,1133


In [9]:
# Count pandas-recognized missing cells; each True contributes 1.
print("Missing cells:", train_df.isna().sum().sum())

# Check IDs and full rows for repeats.
print("Repeated IDs:", train_df["id"].duplicated().sum())

print("Exact duplicate rows:", train_df.duplicated().sum())

# Ignore ID to reveal repeated measurements and annotations.
print(
    "Repeated rows when ID is excluded:",
    train_df.drop(columns="id").duplicated().sum(),
)

Missing cells: 0
Repeated IDs: 0
Exact duplicate rows: 0
Repeated rows when ID is excluded: 67601


In [10]:
# Count distinct values per column and show the smallest counts first.
train_df.nunique(dropna=False).sort_values()

is_sm_ips_ports           2
label                     2
is_ftp_login              4
ct_ftp_cmd                4
ct_state_ttl              5
dttl                      6
dwin                      7
state                     9
attack_cat               10
ct_flw_http_mthd         11
trans_depth              11
sttl                     11
service                  13
swin                     13
ct_dst_sport_ltm         32
ct_src_dport_ltm         47
ct_dst_ltm               50
ct_src_ltm               50
ct_srv_dst               52
ct_srv_src               52
ct_dst_src_ltm           54
proto                   133
dloss                   370
sloss                   409
dpkts                   443
spkts                   480
dmean                  1328
smean                  1357
response_body_len      2386
dbytes                 6660
sbytes                 7214
ackdat                37708
synack                40142
tcprtt                43319
dur                   74039
dinpkt              

In [11]:
# Summarize numeric ranges and spread; .T puts each feature on a row.
# 50% is the median; a much larger mean suggests a long upper tail.
train_df.describe().T

,count,mean,std,min,25%,50%,75%,max
id,175341.0,8.767100e+04,5.061673e+04,1.0,43836.000000,87671.000000,1.315060e+05,1.753410e+05
dur,175341.0,1.359389e+00,6.480249e+00,0.0,0.000008,0.001582,6.680690e-01,5.999999e+01
spkts,175341.0,2.029866e+01,1.368876e+02,1.0,2.000000,2.000000,1.200000e+01,9.616000e+03
dpkts,175341.0,1.896959e+01,1.102583e+02,0.0,0.000000,2.000000,1.000000e+01,1.097400e+04
sbytes,175341.0,8.844844e+03,1.747656e+05,28.0,114.000000,430.000000,1.418000e+03,1.296523e+07
dbytes,175341.0,1.492892e+04,1.436542e+05,0.0,0.000000,164.000000,1.102000e+03,1.465555e+07
rate,175341.0,9.540619e+04,1.654010e+05,0.0,32.786140,3225.806520,1.250000e+05,1.000000e+06
sttl,175341.0,1.795470e+02,1.029400e+02,0.0,62.000000,254.000000,2.540000e+02,2.550000e+02
dttl,175341.0,7.960957e+01,1.105069e+02,0.0,0.000000,29.000000,2.520000e+02,2.540000e+02
sload,175341.0,7.345403e+07,1.883574e+08,0.0,13053.338870,879674.750000,8.888889e+07,5.988000e+09


In [12]:
# Inspect FTP values: the documented binary field also contains 2 and 4.
print("is_ftp_login:")
print(train_df["is_ftp_login"].value_counts(dropna=False).sort_index())

print("\nct_ftp_cmd:")
print(train_df["ct_ftp_cmd"].value_counts(dropna=False).sort_index())

is_ftp_login:
is_ftp_login
0    172774
1      2545
2         6
4        16
Name: count, dtype: int64

ct_ftp_cmd:
ct_ftp_cmd
0    172774
1      2545
2         6
4        16
Name: count, dtype: int64


In [13]:
# Compare FTP columns row by row; matching value counts alone are insufficient.
ftp_columns_match = train_df["is_ftp_login"].eq(train_df["ct_ftp_cmd"])

# True counts a match; ~ reverses the mask to count differences.
print("Matching rows:", ftp_columns_match.sum())
print("Different rows:", (~ftp_columns_match).sum())

Matching rows: 175341
Different rows: 0


In [14]:
# Exclude the ID, answer-revealing category, and target from candidate inputs.
candidate_inputs = train_df.drop(columns=["id", "attack_cat", "label"])

print("Candidate input columns:", candidate_inputs.shape[1])
# duplicated() counts repeats after the first occurrence of each input pattern.
print("Repeated input patterns:", candidate_inputs.duplicated().sum())

Candidate input columns: 42
Repeated input patterns: 74301


In [15]:
# Use all 42 candidate input columns as the keys for matching rows.
input_columns = candidate_inputs.columns.tolist()

# Summarize each input pattern: row count and distinct annotation counts.
pattern_summary = train_df.groupby(
    input_columns,
    dropna=False,
    sort=False,
).agg(
    records=("label", "size"),
    binary_labels=("label", "nunique"),
    attack_categories=("attack_cat", "nunique"),
)

In [16]:
# Mark groups with more than one binary label or attack-category annotation.
binary_conflicts = pattern_summary["binary_labels"] > 1
category_conflicts = pattern_summary["attack_categories"] > 1

print("Distinct input patterns:", len(pattern_summary))

print(
    "Patterns containing both binary labels:",
    binary_conflicts.sum(),
)

# .loc selects conflicting groups; add their record counts to count rows.
print(
    "Total rows in those binary-conflicting patterns:",
    pattern_summary.loc[binary_conflicts, "records"].sum(),
)

print(
    "Patterns containing multiple attack categories:",
    category_conflicts.sum(),
)

Distinct input patterns: 101040
Patterns containing both binary labels: 229
Total rows in those binary-conflicting patterns: 940
Patterns containing multiple attack categories: 1772


## Initial findings

- The training file contains 175,341 rows and 45 columns.
- The target mapping is verified: 0 = normal, 1 = attack.
- Normal traffic accounts for 31.94% of records; attacks account for 68.06%.
- `attack_cat` reveals the binary target and must be excluded from model inputs.
- Pandas detects no missing cells. Special placeholder values still need
  interpretation using the feature documentation.
- IDs are unique. Excluding ID reveals 67,601 repeated rows.
- Excluding ID and target annotations leaves 42 candidate input columns,
  with 74,301 repetitions and 101,040 distinct input patterns.
- 229 input patterns contain both binary labels, involving 940 records.
- 1,772 input patterns contain multiple attack-category annotations.
- `is_ftp_login` and `ct_ftp_cmd` match in every training row.
- `is_ftp_login` contains 22 values outside its documented binary range.
  Their meaning remains unresolved.
- Some numerical variables have means far above their medians,
  suggesting skewed distributions that warrant investigation.

## Implications for later work

- Preserve the original data while deciding how to handle these findings.
- Account for repeated input patterns when designing validation.
- Investigate ambiguous feature values before transforming them.
- Compare against a simple baseline and evaluate more than accuracy.
- Keep the official test set reserved for final evaluation.

## Network feature understanding

The goal is to connect each measurement to network behavior before choosing plots or preprocessing.

- **Protocol** describes the communication rules.
- **Service** identifies application-level activity.
- **State** summarizes the observed communication status according to the protocol and extraction tool.

For example, a record can identify `tcp` as its protocol and `http` as its service. Protocol, service, and state provide context; none alone establishes whether traffic is malicious.

### Selected features

Definitions are based on the [downloaded feature dictionary](../data/raw/NUSW-NB15_features.csv). The reasons a feature might help are hypotheses to investigate, and preprocessing choices remain provisional.

| Feature | Type | Meaning | Why it might help | Potential preprocessing |
|---|---|---|---|---|
| `proto` | Categorical text | Protocol recorded for the transaction | Expected behavior varies by protocol | Categorical encoding |
| `service` | Categorical text | Identified service, such as HTTP, DNS, or FTP | Services have different traffic patterns | Categorical encoding |
| `state` | Categorical text | Recorded protocol-dependent state | Communication progress or termination provides context | Categorical encoding |
| `dur` | Numerical, floating point | Total duration of the record; unit unconfirmed | Brief and sustained activity may have different patterns | Consider scaling for models that benefit from it |
| `spkts` | Numerical count | Packets from source to destination | Measures source-direction activity | Preserve as a count; consider scaling |
| `dpkts` | Numerical count | Packets from destination to source | Measures reverse-direction activity | Preserve as a count; consider scaling |
| `sbytes` | Numerical count | Recorded bytes from source to destination | Measures volume in that direction | Preserve numerically; inspect skew |
| `dbytes` | Numerical count | Recorded bytes from destination to source | Measures volume in the reverse direction | Preserve numerically; inspect skew |

The dictionary spells the packet fields `Spkts` and `Dpkts`; the prepared CSV uses lowercase names.

**Packet count and byte count measure different things.** Three packets with recorded sizes of 100, 200, and 300 bytes give `spkts = 3` and `sbytes = 600`. The byte total should not be assumed to represent application payload alone: Argus documents transaction bytes separately from application bytes.

A small request can produce a large response, so different byte totals in the two directions can occur in legitimate traffic.

### Service names observed in the training file

These are general protocol roles. They do not establish precisely how the historical dataset extractor assigned each label.

| Service | General role |
|---|---|
| `dhcp` | Supplies network configuration, including IP-address assignments |
| `dns` | Queries domain-name information, including address records |
| `ftp` | FTP commands and control communication |
| `ftp-data` | FTP data transfers |
| `http` | Web requests and responses |
| `irc` | Internet Relay Chat |
| `pop3` | Retrieves email from a server |
| `smtp` | Sends and relays email |
| `radius` | Supports authentication and authorization for network access |
| `snmp` | Monitors and manages network devices |
| `ssh` | Provides secure remote access |
| `ssl` | Label associated with SSL/TLS traffic; the version and enclosed application are not established by this value |
| `-` | Placeholder whose precise dataset-specific meaning remains unclear |

The [IANA registry](https://www.iana.org/assignments/service-names-port-numbers) provides standard service names and protocol references. [Zeek's SSL/TLS documentation](https://docs.zeek.org/en/current/reference/logs/ssl.html) explains that its `ssl` label also applies to TLS traffic.

### State codes observed in the training file

The meanings below use the current [Argus manual](https://raw.githubusercontent.com/openargus/clients/main/man/man1/ra.1) as supporting context. The exact historical export behavior can depend on the extraction version and configuration.

| State | Meaning or interpretation limit |
|---|---|
| `CON` | An established or continuing transaction |
| `INT` | Initial transaction report for connectionless protocols such as UDP |
| `REQ` | A TCP connection request |
| `RST` | A reset TCP transaction |
| `ECO` | ICMP echo request |
| `PAR` | ICMP parameter-problem message |
| `URN` | ICMP network-unreachable message |
| `FIN` | TCP FIN signals that a sender has finished sending data; the precise condition producing this dataset's summary code remains unverified |
| `no` | Dataset-specific meaning unverified; preserve the literal category |

The [TCP specification](https://www.rfc-editor.org/rfc/rfc9293.html) documents FIN behavior. A state describes communication behavior, while `label` is a separate normal/attack annotation. Resets and unreachable-network messages can occur during ordinary network failures.

### Documentation uncertainties and processing implications

- The downloaded dictionary describes `dur` as total duration without specifying its unit. Confirm the unit before labeling plots with seconds or another unit.
- The exact meaning of `service="-"` remains unclear. Preserve the value while investigating; it is currently a string rather than a pandas missing value.
- The meaning of `state="no"` and the precise export condition for `state="FIN"` remain unverified.
- Categorical encoding and numerical scaling are possible later preprocessing steps. These notes do not transform the data or finalize the feature set.
- Protocol, service, and state should be interpreted together with other measurements.

Dataset provenance and the original papers are linked from the [official UNSW-NB15 page](https://research.unsw.edu.au/projects/unsw-nb15-dataset).

In [17]:
# Inspect a small set of documented features before planning preprocessing.
selected_features = [
    "proto",
    "service",
    "state",
    "dur",
    "spkts",
    "dpkts",
    "sbytes",
    "dbytes",
]

# Display eight reproducible examples; the full table is unchanged.
train_df[selected_features].sample(n=8, random_state=42)

,proto,service,state,dur,spkts,dpkts,sbytes,dbytes
15482,tcp,-,FIN,2.736664,232,438,13350,548216
133349,udp,dns,INT,0.000009,2,0,114,0
80485,tcp,-,FIN,5.788526,36,34,6102,3892
29972,tcp,-,FIN,3.849634,448,858,25160,1094788
18339,udp,dns,CON,0.001052,2,2,130,162
170500,udp,dns,INT,0.000005,2,0,114,0
165830,udp,dns,INT,0.000008,2,0,114,0
55215,tcp,-,FIN,1.465899,34,16,28660,814


In [18]:
# List category names so placeholders and unfamiliar codes can be documented.
print("Services:", sorted(train_df["service"].unique()))
print("States:", sorted(train_df["state"].unique()))

Services: ['-', 'dhcp', 'dns', 'ftp', 'ftp-data', 'http', 'irc', 'pop3', 'radius', 'smtp', 'snmp', 'ssh', 'ssl']
States: ['CON', 'ECO', 'FIN', 'INT', 'PAR', 'REQ', 'RST', 'URN', 'no']


In [19]:
from sklearn.model_selection import StratifiedGroupKFold

# Give identical 42-feature inputs the same group number for splitting.
# Group numbers are split metadata, not model inputs.
pattern_groups = train_df.groupby(
    input_columns,
    dropna=False,
    sort=False,
).ngroup()

print("Distinct input groups:", pattern_groups.nunique())

Distinct input groups: 101040


In [20]:
# Aim for five folds with similar class proportions, keeping each group together.
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

# Keep the first split: four folds for training, one for validation (~80/20).
fit_indices, validation_indices = next(
    splitter.split(
        X=candidate_inputs,
        y=train_df["label"],
        groups=pattern_groups,
    )
)

# Select returned row positions; train_df remains the full official training data.
fit_df = train_df.iloc[fit_indices].copy()
validation_df = train_df.iloc[validation_indices].copy()

In [21]:
# Collect the unique input groups assigned to each partition.
fit_groups = set(pattern_groups.iloc[fit_indices])
validation_groups = set(pattern_groups.iloc[validation_indices])

# Intersections reveal any groups or records present on both sides.
shared_groups = fit_groups.intersection(validation_groups)
shared_ids = set(fit_df["id"]).intersection(validation_df["id"])

# Stop if there is overlap or if the partitions do not retain every row.
assert len(shared_groups) == 0
assert len(shared_ids) == 0
assert len(fit_df) + len(validation_df) == len(train_df)

print("Shared input groups:", len(shared_groups))
print("Shared IDs:", len(shared_ids))
print("Total rows preserved:", len(fit_df) + len(validation_df))

Shared input groups: 0
Shared IDs: 0
Total rows preserved: 175341


In [22]:
# Compare the size and class proportions of our two partitions.
split_summary = pd.DataFrame(
    {
        "rows": [len(fit_df), len(validation_df)],
        "normal_rows": [
            fit_df["label"].eq(0).sum(),
            validation_df["label"].eq(0).sum(),
        ],
        "attack_rows": [
            fit_df["label"].eq(1).sum(),
            validation_df["label"].eq(1).sum(),
        ],
        "attack_percentage": [
            fit_df["label"].mean() * 100,
            validation_df["label"].mean() * 100,
        ],
    },
    index=["Training", "Validation"],
)

split_summary.round(2)

,rows,normal_rows,attack_rows,attack_percentage
Training,140272,44800,95472,68.06
Validation,35069,11200,23869,68.06


In [23]:
# Create the folder for data derived from the original CSV.
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# Create a table containing each record's ID and assigned partition.
split_assignments = train_df[["id"]].copy()
split_assignments["partition"] = "training"

# Mark IDs belonging to the validation partition.
is_validation = split_assignments["id"].isin(validation_df["id"])
split_assignments.loc[is_validation, "partition"] = "validation"

# Save the assignments without adding the pandas row index.
split_path = processed_dir / "train_validation_split.csv"
split_assignments.to_csv(split_path, index=False)

print("Saved:", split_path)
print(split_assignments["partition"].value_counts())

Saved: C:\Code\fourthyear\resume_projects\network_intrusion_detection_system\data\processed\train_validation_split.csv
partition
training      140272
validation     35069
Name: count, dtype: int64
